In [15]:
import os
import time
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai import errors



In [10]:
load_dotenv()
client = genai.Client()

### Gemini api call with streaming enabled

In [74]:
config = types.GenerateContentConfig(system_instruction="do not provide code",max_output_tokens=4096)
#model = "gemini-3.1-flash-lite"
#model = "gemini-2.5-flash"
model = "gemini-2.5-flash-lite"
prompt = "say hello in 10 random languagsm"

### api call with retry error handling, exponential backoff, and streaming

In [110]:
def call_with_retry_and_streaming(client, max_retries=3, **kwargs):
    for attempt in range(max_retries):
        try:

            response_stream  = client.models.generate_content_stream(**kwargs)

            return response_stream
            
        except errors.APIError as e:
            status_code = e.code if e.code is not None else 0
            
            if status_code == 429 or status_code >= 500:
                wait = 2 ** attempt  # Exponential backoff
                print(f"[Attempt {attempt + 1}] Encountered {status_code}. Retrying in {wait}s...")
                time.sleep(wait)
                
            else:
                print(f"Critical client error {status_code}: {e.message}. Exiting.")
                raise e
                
    raise Exception("Max retries exceeded")

In [ ]:
response_stream = call_with_retry_and_streaming(client = client, max_retries = 3, model = model, contents = prompt, config = config)

In [106]:
for chunk in response_stream:
    print(chunk.text, end="", flush=True)

Here are hellos in 10 languages, chosen somewhat randomly:

1.  **Swahili:** Jambo
2.  **Korean:** 안녕하세요 (Annyeonghaseyo)
3.  **Irish Gaelic:** Dia duit
4.  **Greek:** Γεια σου (Ya sou)
5.  **Tagalog:** Kamusta
6.  **Hungarian:** Szia
7.  **Finnish:** Moi
8.  **Vietnamese:** Xin chào
9.  **Hebrew:** שלום (Shalom)
10. **Turkish:** Merhaba